# Lab 4B – ML Endpoint Testing

Acceptance test for the **Lab 4A** CI/CD deploy: the GitHub Actions `deploy.yml`
workflow in the model-deploy repo, which runs `cdk deploy` and creates a
real-time endpoint from the latest **Approved** version in your project's Model
Package Group.

The notebook proves four things about that endpoint:

- it is the Lab 4A endpoint, and the model behind it came from an `Approved`
  version of your project's Model Package Group;
- it honours both payload formats the shared inference handler advertises;
- it gives sensible answers on the exact evaluation rows the build pipeline
  scored the model on;
- you know what it does with malformed input.

## Prerequisites

- `lab0-setup/setup.ipynb` has run in this kernel. It installs every package this
  notebook needs and writes the repo-root `.env`. This notebook has no
  `pip install` cell, on purpose.
- **Lab 4A is complete:**
  1. You created a SageMaker project from the classification template. Its
     `ModelName` parameter (default `bank-classification`) names everything below.
  2. The build workflow (`build.yml`) ran the SageMaker Pipeline, which
     registered a version in `<ModelName>-model-group`.
  3. You approved that version, and the deploy workflow (`deploy.yml`) finished.
     The endpoint is named **`<ModelName>-model-group-endpoint`** and it is
     `InService`.
- The Lab 2 Iceberg tables exist. The Lab 4A build pipeline already needed them.

This notebook does **not** test the Lab 3A endpoint (`bank-marketing-<timestamp>`).
That endpoint is tested in Lab 3A, Section 6.

## What you'll do

1. Find the Lab 4A endpoint. Nothing to paste in.
2. Trace it back to its Model Registry version, check the approval, and load the
   version's `baseline.json`.
3. Send one JSON prediction and one CSV prediction.
4. Score 50 rows of the pinned evaluation snapshot and compare against the
   baseline metrics.
5. Check how the endpoint handles malformed payloads.

## Step 1: Setup and endpoint discovery

In [ ]:
# The ModelName you gave the Lab 4A SageMaker project (template default:
# "bank-classification"). Leave it blank to use the most recently updated Lab 4A
# endpoint in this account and Region.
MODEL_NAME = ""

In [ ]:
import io
import json
import os
import sys
import time
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError

# The repo-root .env written by lab0-setup is the single source of truth for
# project naming; workshop_common is the single source for the feature list.
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / "workshop_env.py").exists():
        sys.path.insert(0, str(_cand))
        break
else:
    raise RuntimeError("Could not locate the repo root (no workshop_env.py found).")

from workshop_common.env import load_workshop_env
from workshop_common.schema import feature_names, target_column

env = load_workshop_env()
REGION = os.environ.get("AWS_DEFAULT_REGION") or boto3.Session().region_name
ATHENA_DATABASE = env["ATHENA_DATABASE"]
FEATURES = feature_names()
TARGET = target_column()

sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

print(f"Region        : {REGION}")
print(f"Athena DB     : {ATHENA_DATABASE}")
print(f"Features ({len(FEATURES)}): {', '.join(FEATURES)}")
print(f"Target        : {TARGET}")

The Lab 4A deploy stack names the endpoint after the Model Package Group it
deploys from: `f"{MODEL_PACKAGE_GROUP_NAME}-endpoint"`
(`model_deploy/deploy_endpoint/deploy_endpoint_stack.py`). The project template
sets the group to `<ModelName>-model-group`, so the endpoint is always
`<ModelName>-model-group-endpoint`. The cell below uses that naming rule. It
never picks up a Lab 3A `bank-marketing-*` endpoint by accident.

The endpoint name stays the same across deploys: each approval updates the
endpoint in place. So the newest endpoint is the one with the latest
`LastModifiedTime`, not the latest `CreationTime`.

In [ ]:
ENDPOINT_SUFFIX = "-model-group-endpoint"


def find_lab4a_endpoint():
    # 1. An explicit ENDPOINT_NAME (shell or .env) wins, but only if it is a Lab
    #    4A endpoint. Lab 5D also reads ENDPOINT_NAME from .env, so it may point
    #    at a Lab 3A endpoint; that is ignored here rather than tested.
    explicit = os.environ.get("ENDPOINT_NAME", "").strip()
    if explicit:
        if explicit.endswith(ENDPOINT_SUFFIX):
            return explicit, "ENDPOINT_NAME"
        print(f"Ignoring ENDPOINT_NAME={explicit!r}: it does not end with "
              f"'{ENDPOINT_SUFFIX}', so it is not a Lab 4A endpoint.")
    # 2. The project's ModelName gives the name directly.
    if MODEL_NAME.strip():
        return f"{MODEL_NAME.strip()}{ENDPOINT_SUFFIX}", "MODEL_NAME"
    # 3. Otherwise, the most recently updated Lab 4A endpoint.
    candidates = []
    for page in sm.get_paginator("list_endpoints").paginate(NameContains=ENDPOINT_SUFFIX):
        candidates += [e for e in page["Endpoints"]
                       if e["EndpointName"].endswith(ENDPOINT_SUFFIX)]
    if not candidates:
        raise RuntimeError(
            f"No endpoint named '*{ENDPOINT_SUFFIX}' in {REGION}. Finish Lab 4A first: "
            "approve a version in your project's Model Package Group and wait for "
            "the deploy.yml GitHub Actions run to succeed. Then re-run this cell.")
    candidates.sort(key=lambda e: e["LastModifiedTime"], reverse=True)
    if len(candidates) > 1:
        print("Several Lab 4A endpoints found. Using the most recently updated one.")
        print("Set MODEL_NAME above to choose a different one:")
        for e in candidates:
            print(f"  {e['EndpointName']:55s} {e['EndpointStatus']:12s} "
                  f"updated {e['LastModifiedTime']:%Y-%m-%d %H:%M}")
    return candidates[0]["EndpointName"], "auto-discovered"


endpoint_name, found_by = find_lab4a_endpoint()
MODEL_PACKAGE_GROUP = endpoint_name[: -len("-endpoint")]

try:
    desc = sm.describe_endpoint(EndpointName=endpoint_name)
except ClientError as exc:
    raise RuntimeError(
        f"Endpoint {endpoint_name} does not exist. Check MODEL_NAME, and check that "
        "the deploy.yml GitHub Actions run for this model succeeded.") from exc

print(f"Endpoint       : {endpoint_name}  ({found_by})")
print(f"Package group  : {MODEL_PACKAGE_GROUP}")
print(f"Status         : {desc['EndpointStatus']}")
print(f"Created        : {desc['CreationTime']:%Y-%m-%d %H:%M:%S}")
print(f"Last updated   : {desc['LastModifiedTime']:%Y-%m-%d %H:%M:%S}")
print(f"Endpoint config: {desc['EndpointConfigName']}")
for variant in desc.get("ProductionVariants", []):
    print(f"  variant {variant['VariantName']}: "
          f"{variant.get('CurrentInstanceCount')} instance(s), "
          f"weight {variant.get('CurrentWeight')}")

# The CDK stack that owns the endpoint. Needed for clean-up at the end.
_tags = {t["Key"]: t["Value"]
         for t in sm.list_tags(ResourceArn=desc["EndpointArn"]).get("Tags", [])}
DEPLOY_STACK = _tags.get("aws:cloudformation:stack-name", "")
print(f"Deploy stack   : {DEPLOY_STACK or '(no CloudFormation tag found)'}")

if desc["EndpointStatus"] != "InService":
    raise RuntimeError(
        f"Endpoint is {desc['EndpointStatus']}, not InService. "
        + ("A deploy is still rolling out: wait for the deploy.yml run to finish, "
           "then re-run this cell." if desc["EndpointStatus"] in ("Creating", "Updating")
           else f"FailureReason: {desc.get('FailureReason', 'n/a')}"))

## Step 2: Trace the endpoint back to the Model Registry

The deploy stage's governance claim is that *only an approved model reaches an
endpoint*. This walks the chain that proves it for the running endpoint:

`endpoint → endpoint config → model → model package → approval status`

It also checks two things specific to the Lab 4A deploy:

- **Is the deployed version the newest approved one?** `get_approved_package.py`
  picks the version to deploy: the newest approved one in the group. The cell
  below warns when the endpoint serves an older version. That happens while a
  newer approval's deploy is still running, if that deploy failed, or with
  deploy code from before the `SortOrder="Descending"` fix (it took the
  **oldest** approved version).
- **Is the handler fully wired?** The deploy reads `FEATURE_NAMES` from the
  version's `baseline.json` and sets it on the container next to the script-mode
  flags. Without `SAGEMAKER_PROGRAM`, the stock XGBoost container ignores the
  custom handler. Without `FEATURE_NAMES`, a CSV request can't be mapped to
  named features.

In [ ]:
config = sm.describe_endpoint_config(EndpointConfigName=desc["EndpointConfigName"])
model_name = config["ProductionVariants"][0]["ModelName"]
model = sm.describe_model(ModelName=model_name)
container = model.get("PrimaryContainer") or model["Containers"][0]
package_ref = container.get("ModelPackageName")

print(f"Endpoint config: {desc['EndpointConfigName']}")
print(f"Model          : {model_name}")
assert package_ref, (
    "The model was created from a raw artifact URI, not a Model Registry version, "
    "so this is not a Lab 4A deploy (the CDK stack always uses ModelPackageName).")

package = sm.describe_model_package(ModelPackageName=package_ref)
package_arn = package["ModelPackageArn"]
print(f"Package group  : {package['ModelPackageGroupName']}")
print(f"Version        : {package['ModelPackageVersion']}")
print(f"Approval status: {package['ModelApprovalStatus']}")
for key, value in (package.get("CustomerMetadataProperties") or {}).items():
    print(f"  metadata     : {key} = {value}")

assert package["ModelPackageGroupName"] == MODEL_PACKAGE_GROUP, (
    f"The endpoint serves a version from {package['ModelPackageGroupName']}, not "
    f"{MODEL_PACKAGE_GROUP}. Lab 4A always deploys from its own group.")
assert package["ModelApprovalStatus"] == "Approved", (
    f"The endpoint serves a version that is now '{package['ModelApprovalStatus']}'. "
    "Either it was un-approved after the deploy, or the approval gate was bypassed.")
print("\n✓ The endpoint serves an Approved version of the project's Model Package Group")

latest = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP, ModelApprovalStatus="Approved",
    SortBy="CreationTime", SortOrder="Descending", MaxResults=1,
)["ModelPackageSummaryList"]
if latest and latest[0]["ModelPackageArn"] != package_arn:
    print(f"\n⚠ The endpoint serves version {package['ModelPackageVersion']}, but the "
          f"newest approved version is {latest[0]['ModelPackageVersion']}.")
    print("  Check that the deploy.yml run for the newer approval finished. If it")
    print("  did, your deploy repo predates the fix that makes get_approved_package.py")
    print("  pass SortOrder='Descending' to list_model_packages.")
else:
    print("✓ It is the newest approved version in the group")

In [ ]:
# The effective container environment: the version's InferenceSpecification,
# overridden by anything the deploy stack set on the model's container.
_spec_containers = (package.get("InferenceSpecification") or {}).get("Containers") or [{}]
effective_env = {**_spec_containers[0].get("Environment", {}),
                 **container.get("Environment", {})}

print(f"SAGEMAKER_PROGRAM        : {effective_env.get('SAGEMAKER_PROGRAM', '(not set)')}")
print(f"ENABLE_INFERENCE_CAPTURE : {effective_env.get('ENABLE_INFERENCE_CAPTURE', '(not set)')}")
print(f"SQS_QUEUE_URL            : {effective_env.get('SQS_QUEUE_URL') or '(not set)'}")

assert effective_env.get("SAGEMAKER_PROGRAM") == "inference.py", (
    "SAGEMAKER_PROGRAM is not inference.py, so the stock XGBoost container ignores "
    "the custom handler and answers with bare probabilities. The rest of this "
    "notebook expects the handler's JSON response.")

_feature_names_env = [c.strip() for c in effective_env.get("FEATURE_NAMES", "").split(",") if c.strip()]
CSV_SUPPORTED = _feature_names_env == FEATURES
if not _feature_names_env:
    print("\n⚠ FEATURE_NAMES is not set. The deploy could not read the version's")
    print("  baseline.json. Positional CSV requests will be scored on all-zero")
    print("  features, so Step 3 skips the CSV test.")
elif not CSV_SUPPORTED:
    print("\n⚠ FEATURE_NAMES on the endpoint does not match the workshop schema:")
    print(f"  endpoint: {_feature_names_env}")
    print(f"  schema  : {FEATURES}")
else:
    print(f"FEATURE_NAMES            : {len(_feature_names_env)} features, schema order ✓")

capture_on = (effective_env.get("ENABLE_INFERENCE_CAPTURE") == "true"
              and bool(effective_env.get("SQS_QUEUE_URL")))
print(f"\nInference capture is {'ON' if capture_on else 'OFF'}: every request in this "
      f"notebook {'is' if capture_on else 'is not'} written to inference_responses.")

In [ ]:
# baseline.json is what the build pipeline's evaluation step wrote for THIS
# version: its metrics, and the Iceberg snapshot of the rows it scored.
# The register step links it as ModelMetrics.ModelQuality.Statistics.
baseline = {}
baseline_uri = (((package.get("ModelMetrics") or {}).get("ModelQuality") or {})
                .get("Statistics") or {}).get("S3Uri")
if baseline_uri:
    _bucket, _key = baseline_uri.removeprefix("s3://").split("/", 1)
    try:
        baseline = json.loads(boto3.client("s3", region_name=REGION)
                              .get_object(Bucket=_bucket, Key=_key)["Body"].read())
    except ClientError as exc:
        print(f"⚠ Could not read {baseline_uri}: {exc.response['Error']['Code']}")
else:
    print("⚠ This version has no ModelQuality statistics, so there is no baseline.json.")

if baseline:
    print(f"Baseline       : {baseline_uri}")
    print(f"Evaluated on   : {baseline.get('evaluation_table')} "
          f"@ snapshot {baseline.get('evaluation_snapshot_id') or '(unpinned)'}")
    print(f"Sample size    : {baseline.get('sample_size')} rows "
          f"({baseline.get('positive_samples')} positive)")
    for name, value in (baseline.get("metrics") or {}).items():
        print(f"  {name:10s} {value:.4f}")
    if baseline.get("feature_schema") and baseline["feature_schema"] != FEATURES:
        print("\n⚠ baseline.json's feature_schema differs from the workshop schema.")

## Step 3: Single prediction — both payload formats

The shared inference handler (`model_build/source_scripts/inference/inference.py`)
accepts two content types and always answers with the same JSON:

```json
{"predictions": [0], "probabilities": {"yes": [0.07], "no": [0.93]}}
```

- `application/json`: an object of **named** features. Order doesn't matter.
- `text/csv`: one row with no header and **positional** values, in schema order.
  The handler names the columns using `FEATURE_NAMES`.

The same client is sent both ways, so the two answers must agree.

In [ ]:
def invoke(payload, content_type="application/json"):
    # Invoke the endpoint and return (parsed_response, latency_ms).
    body = json.dumps(payload) if content_type == "application/json" else payload
    started = time.perf_counter()
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name, ContentType=content_type,
        Accept="application/json", Body=body,
    )
    latency_ms = (time.perf_counter() - started) * 1000
    return json.loads(response["Body"].read().decode()), latency_ms


def positive_probability(result):
    # The handler returns P(yes) under probabilities["yes"].
    return result["probabilities"]["yes"][0]


# A representative client, on the same label-encoded scale as the training data.
sample_client = {
    "age": 41, "job": 0, "marital": 1, "education": 6, "credit_default": 0,
    "housing": 2, "loan": 0, "contact": 1, "month": 6, "day_of_week": 1,
    "duration": 180, "campaign": 2, "pdays": 999, "previous": 0,
    "poutcome": 1, "emp_var_rate": 1.1, "cons_price_idx": 93.994,
    "cons_conf_idx": -36.4, "euribor3m": 4.857, "nr_employed": 5191.0,
}
json_payload = {f: sample_client[f] for f in FEATURES}

json_result, json_latency = invoke(json_payload)
print("JSON payload")
print(f"  raw response : {json_result}")

# The response shape is the contract Lab 5 and any client depend on.
assert "predictions" in json_result and "probabilities" in json_result, (
    "Response is missing 'predictions' or 'probabilities': the custom handler is "
    "not running.")
print(f"  prediction   : {json_result['predictions'][0]}")
print(f"  P({TARGET}=1) : {positive_probability(json_result):.4f}")
print(f"  latency      : {json_latency:.0f} ms")
print("\n✓ The response matches the documented format")

In [ ]:
# Same client as one positional CSV row, in schema order.
if not CSV_SUPPORTED:
    print("Skipped: FEATURE_NAMES on the endpoint does not match the schema (see Step 2),")
    print("so a positional CSV row would not be mapped to the right features.")
else:
    csv_row = ",".join(str(json_payload[f]) for f in FEATURES)
    csv_result, csv_latency = invoke(csv_row, content_type="text/csv")
    print(f"CSV payload  : {csv_row}")
    print(f"  prediction : {csv_result['predictions'][0]}")
    print(f"  P({TARGET}=1) : {positive_probability(csv_result):.4f}")
    print(f"  latency    : {csv_latency:.0f} ms")

    # The same features in either format must give the same score.
    assert csv_result["predictions"][0] == json_result["predictions"][0]
    assert abs(positive_probability(csv_result) - positive_probability(json_result)) < 1e-6
    print("\n✓ JSON and CSV payloads agree, so the feature order is correct")

## Step 4: Score the pinned evaluation rows

This sends 50 rows, one request each, from the **same Iceberg snapshot** the build
pipeline evaluated this version on. `baseline.json` records it as
`evaluation_snapshot_id`. Athena's `FOR VERSION AS OF` reads that snapshot, so
later writes to `evaluation_data` don't change the rows.

This is a smoke test, not a benchmark. It catches an endpoint that answers but
answers badly: errors, multi-second latency, or every row getting the same
class. Agreement on 50 rows is a rough signal only. The build pipeline's
metrics, printed in Step 2, cover the full evaluation set.

If the Athena read fails, the cell falls back to 50 copies of the sample client
with random `age` and `duration` values, and skips the agreement check.

In [ ]:
import awswrangler as wr

N_REQUESTS = 50

eval_table = baseline.get("evaluation_table") or env["EVALUATION_TABLE"]
snapshot_id = baseline.get("evaluation_snapshot_id", "")
source = f"{eval_table} FOR VERSION AS OF {snapshot_id}" if snapshot_id else eval_table

try:
    # Same results location the build pipeline uses: the bucket behind the
    # Glue database.
    _db = boto3.client("glue", region_name=REGION).get_database(Name=ATHENA_DATABASE)["Database"]
    athena_output = f"s3://{_db['LocationUri'].split('/')[2]}/athena-query-results/"
    # No boto3_session argument: Studio has Ray installed, which puts awswrangler
    # in distributed mode, and that mode rejects boto3_session. The default
    # session already uses this notebook's Region (the same call as Lab 3A).
    sample = wr.athena.read_sql_query(
        f"SELECT CAST({TARGET} AS integer) AS y, {', '.join(FEATURES)} "
        f"FROM {source} ORDER BY rand() LIMIT {N_REQUESTS}",
        database=ATHENA_DATABASE, s3_output=athena_output, ctas_approach=False,
    )
    # Distributed mode returns a Modin DataFrame; 50 rows are fine in pandas.
    if not isinstance(sample, pd.DataFrame):
        sample = sample._to_pandas()
    actuals = [int(v) for v in sample["y"]]
    print(f"Read {len(sample)} rows from {ATHENA_DATABASE}.{source}")
    if not snapshot_id:
        print("⚠ baseline.json has no evaluation_snapshot_id, so these rows come from")
        print("  the live table, not necessarily the rows the model was evaluated on.")
except Exception as exc:
    print(f"⚠ Could not read {ATHENA_DATABASE}.{source} "
          f"({exc.__class__.__name__}: {str(exc)[:200]})")
    print("  Falling back to perturbed copies of the sample client.")
    rng = np.random.default_rng(42)
    sample = pd.DataFrame([
        {**json_payload, "age": int(rng.integers(20, 70)),
         "duration": int(rng.integers(30, 900))}
        for _ in range(N_REQUESTS)
    ])
    actuals = None


def as_number(value):
    # JSON has no NaN; send nulls and let the handler coerce them.
    return None if pd.isna(value) else float(value)


latencies, predictions, probabilities, errors = [], [], [], []
for _, row in sample.iterrows():
    try:
        result, latency = invoke({f: as_number(row[f]) for f in FEATURES})
        latencies.append(latency)
        predictions.append(int(result["predictions"][0]))
        probabilities.append(positive_probability(result))
    except Exception as exc:
        errors.append(f"{exc.__class__.__name__}: {exc}")

print(f"\nSent            : {len(sample)}  ({len(errors)} errors)")
assert not errors, f"{len(errors)} invocation(s) failed, first ones: {errors[:3]}"

lat = np.array(latencies)
print(f"Latency ms      : p50 {np.percentile(lat, 50):.0f} | "
      f"p90 {np.percentile(lat, 90):.0f} | max {lat.max():.0f}")
print(f"Predicted {TARGET}=1 : {sum(predictions)}/{len(predictions)} "
      f"({sum(predictions) / len(predictions) * 100:.1f}%)")
print(f"P({TARGET}=1)     : min {min(probabilities):.4f} | "
      f"median {np.median(probabilities):.4f} | max {max(probabilities):.4f}")

assert set(predictions) <= {0, 1}, f"Unexpected prediction values: {set(predictions)}"
if len(set(np.round(probabilities, 6))) == 1:
    print("\n⚠ Every row got the same probability. The model is ignoring its input:")
    print("  check the feature mapping, not the endpoint.")

if actuals is not None:
    correct = sum(int(p == a) for p, a in zip(predictions, actuals))
    print(f"\nAgreement with labels: {correct}/{len(actuals)} "
          f"({correct / len(actuals) * 100:.1f}%)")
    if baseline.get("metrics", {}).get("accuracy") is not None:
        print(f"Build pipeline accuracy on the full set: "
              f"{baseline['metrics']['accuracy'] * 100:.1f}%")
    print("(50 rows is a rough signal only. Expect a few points either way.)")

## Step 5: Malformed payload behaviour

You should know what your endpoint does with bad input, and the answer is not
"it rejects it". The handler is built to avoid failing requests:

- Values it can't parse as numbers become `NaN`, and XGBoost treats them as
  missing.
- Any expected feature missing from the request is set to `0.0`.
- A CSV row only gets named columns when it has exactly as many values as
  `FEATURE_NAMES`. A shorter or longer row keeps its numbered columns. So
  **every** feature counts as missing and is set to `0.0`. The values you sent are
  dropped completely.

So a short or garbled CSV row still returns `200`, with the same prediction as an
all-zero client. The cell sends that all-zero client first so you can compare.
None of this shows up in the endpoint's error metrics. Lab 5B looks at this
problem. The places to catch it are input validation before the endpoint, or a
data-quality check on the captured features.

If inference capture is on (Step 2), these requests are written to
`inference_responses` like any other. That's a handful of all-zero rows in Lab 5's
data. You'll see them, but they won't move drift metrics computed over thousands
of rows.

In [ ]:
zero_result, _ = invoke({f: 0.0 for f in FEATURES})
p_zero = positive_probability(zero_result)
print(f"{'all-zero client (reference)':30s} -> 200 OK   P={p_zero:.4f}\n")

probes = [
    ("too few CSV values", "56,1,1,1,0", "text/csv"),
    ("non-numeric CSV values", "bad,data,wrong,feature,count", "text/csv"),
    ("unknown JSON feature names", {"not_a_feature": 1.0, "age": 41}, "application/json"),
    ("empty body", "", "text/csv"),
]
for label, payload, content_type in probes:
    try:
        result, latency = invoke(payload, content_type=content_type)
        p = positive_probability(result)
        same = "  (same as all-zero)" if abs(p - p_zero) < 1e-6 else ""
        print(f"{label:30s} -> 200 OK   P={p:.4f}  ({latency:.0f} ms){same}")
    except ClientError as exc:
        err = exc.response["Error"]
        print(f"{label:30s} -> {err['Code']}: {err['Message'][:90]}")
    except Exception as exc:
        print(f"{label:30s} -> {exc.__class__.__name__}: {str(exc)[:90]}")

print("\nTakeaway: most malformed input is absorbed, not rejected. Validate before")
print("the endpoint if a wrongly shaped request should fail visibly.")

## Summary

What this run established for the Lab 4A endpoint:

- It was **found by the Lab 4A naming rule** (`<ModelName>-model-group-endpoint`),
  so a Lab 3A endpoint in the same account is never tested by mistake.
- It serves an **`Approved` version of the project's own Model Package Group**.
  Step 2 warns when the deploy picked an older approved version than the newest
  one.
- **Both payload formats** return the documented response and agree with each
  other, as long as the deploy set `FEATURE_NAMES`.
- **50 rows from the pinned evaluation snapshot** were scored with no errors. The
  notebook reported their latency percentiles and checked their agreement with
  the labels against the build pipeline's own metrics.
- **Malformed input is absorbed, not rejected.** Carry that finding into Lab 5;
  it isn't a bug to fix at the endpoint.

### Going on to Lab 5

Lab 5 monitors the **Lab 3A** endpoint (`bank-marketing-<timestamp>`), not this
one. Keep the Lab 3A endpoint running until you finish Lab 5. If it is gone,
Lab 5A and 5B fall back to any `InService` endpoint, which may be this one, so
redeploy from the Lab 3A notebook rather than relying on that.

### Clean-up

This endpoint isn't needed after this lab, and it bills until it is deleted. Lab
4A created it with CDK, so delete the CloudFormation stack printed in Step 1
(`<ModelName>-model-group-endpoint-stack`; deploy repos created before that
naming change use `sagemaker-endpoint-stack`).
Don't delete the endpoint directly: that leaves the stack out of sync with the
real resources, and the next deploy fails.

```bash
aws cloudformation delete-stack --stack-name <stack from Step 1> --region <region>
```

The next approved version re-creates the stack through the deploy workflow.